# Project Assignment: Short Video Recommender System (KuaiRec)

Dataset Source: [Kuairec](https://kuairec.com/)

Arxiv Paper: [KuaiRec: A Fully-observed Dataset and Insights for Evaluating Recommender Systems](https://arxiv.org/pdf/2202.10842)

## Cosine Similarity Model

Cosine similarity is a metric used to measure how similar two vectors are. Users and items can be represented as vectors in a multi-dimensional space, and cosine similarity can be used to find the most similar items to a given item or the most similar users to a given user.

This is a content based recommendation system that uses cosine similarity to recommend items based on their features.

## Dataset import

The server is down, please download from the Google Drive in the given link.

In [ ]:
!wget https://nas.chongminggao.top:4430/datasets/KuaiRec.zip --no-check-certificate
!unzip KuaiRec.zip

--2025-05-09 17:20:44--  https://nas.chongminggao.top:4430/datasets/KuaiRec.zip
Resolving nas.chongminggao.top (nas.chongminggao.top)... 

In [1]:
# Misc
import numpy as np
import pandas as pd
from utils import get_data_path, matrix_cleanup 

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.decomposition import PCA

# Model training
from sklearn.metrics.pairwise import cosine_similarity

# Plot metrics
import plotly.express as px
import plotly.graph_objects as go


# I get my dataset from a Kaggle input
DATA_PATH = get_data_path()

DATA_PATH

'/home/tofeha/ING2/ING2/REMA1/FinalProject_2025_aziz.zeghal/models/../KuaiRec 2.0/data'

# Step 1: Load the dataset

## Small matrix

This table has a density of 99.6%. This means that 99.6% of the entries in the matrix are non-zero, indicating that most users have interacted with most items.

In [2]:
small_matrix = pd.read_csv(f"{DATA_PATH}/small_matrix.csv")

small_matrix = matrix_cleanup(small_matrix)


## Item category encoding

We have the caracteristics of the videos (author_id, video_type...) but this part requires less preprocessing.

For Content-based filtering, we need to use features of the videos (list of tags). We will use a simple one-hot encoding.

In [3]:
# No missing values for this data
item_categories = pd.read_csv(f"{DATA_PATH}/item_categories.csv")

## Item daily features

This dataset is also interesting for content-based filtering.

Mostly composed of textual data, we will use a TF-IDF vectorizer to encode the features of the videos.

In [4]:
item_daily_features = pd.read_csv(f"{DATA_PATH}/item_daily_features.csv", lineterminator='\n')
item_daily_features.fillna(-1, inplace=True)

## User features

In [5]:
user_features = pd.read_csv(f"{DATA_PATH}/user_features.csv", lineterminator='\n')
user_features.fillna(-1, inplace=True)

# Step 2: Feature Engineering

- Create meaningful features from interaction and metadata (e.g., content tags, user activity history)
- Build user-item interaction matrix
- Optionally extract time-based or popularity-based features

## Item category encoding

We have the caracteristics of the videos (author_id, video_type...) but this part requires less preprocessing.

For Content-based filtering, we need to use features of the videos (list of tags). No need for TF-IDF, we will use a simple one-hot encoding.

- **XXX_features** : The Dataframe with all features, used initially
- **XXX_features_map** : The Dataframe with the mapping of the features, filtered with the features we want
- **XXX_features_columns** : The list of features WITHOUT the ids


#### Item categories

In [6]:
# Use MultiLabelBinarizer to manage efficiently the feat column
mlb = MultiLabelBinarizer()

# Transform the feat column to a list (evaluate with python)
item_categories["feat"] = item_categories["feat"].apply(eval)

item_categories = pd.DataFrame(mlb.fit_transform(item_categories["feat"]), 
                  columns=mlb.classes_,
                  index=item_categories["video_id"])


item_categories.reset_index(drop=True, inplace=True)
item_categories[item_categories.columns] = item_categories[item_categories.columns].astype("int16")

item_categories.head(3)


,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


#### Item daily features

We take the oldest data point for a given video_id.

Depending on the complexity, you can choose the number of features.

In [7]:
TEXT_FEATURES = ["video_type", "upload_type"] # "visible_status"
INT_FEATURES = ['video_duration','video_width', 'video_height', 'music_id', 'video_tag_id','show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
                'play_duration', 'complete_play_cnt', 'complete_play_user_num', 'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
                'long_time_play_user_num', 'short_time_play_cnt', 'short_time_play_user_num', 'play_progress']
#  ['comment_stay_duration',
       # 'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       # 'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       # 'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       # 'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       # 'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       # 'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       # 'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt',
       # 'report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num',
       # 'collect_cnt', 'collect_user_num', 'cancel_collect_cnt',
       # 'cancel_collect_user_num']

In [8]:
# Keep the latest date for each video_id
item_daily_features = item_daily_features.loc[item_daily_features.groupby("video_id")["date"].idxmax()].reset_index(drop=True)

# One-hot str features
text_daily_features = item_daily_features[TEXT_FEATURES]
onehotter = OneHotEncoder(handle_unknown="ignore")
onehot_array = onehotter.fit_transform(text_daily_features).toarray()

# Convert to DataFrame
text_daily_features = pd.DataFrame(
    onehot_array,
    columns=onehotter.get_feature_names_out(TEXT_FEATURES),
    index=item_daily_features.index)

# Merge the one-hot encoded features back into the original DataFrame
item_daily_features = pd.concat([item_daily_features[INT_FEATURES], text_daily_features], axis=1)

In [9]:
# No IDs, because it is the index
item_features_map = pd.concat([item_daily_features, item_categories], axis=1)

# Column names should be str
item_features_map.columns = item_features_map.columns.map(str)

# We can keep all the columns
item_features_columns = item_features_map.columns.tolist()

### User tower

In [10]:
user_features_columns = [
    "is_lowactive_period","is_live_streamer", "is_video_author",
    "onehot_feat0", "onehot_feat1", "onehot_feat2", "onehot_feat3",
    "onehot_feat4", "onehot_feat5", "onehot_feat6", "onehot_feat7",
    "onehot_feat8", "onehot_feat9", "onehot_feat10", "onehot_feat11", 
    "onehot_feat12", "onehot_feat13", "onehot_feat14", "onehot_feat15",
    "onehot_feat16", "onehot_feat17"
]
user_features_map = user_features[user_features_columns].copy()

user_features_map[user_features_map.columns] = user_features_map[user_features_map.columns].astype("int16")

In [11]:
# Index is the associated IDs for quick creation
display(user_features_map.head(3))
display(item_features_map.head(3))

,is_lowactive_period,is_live_streamer,is_video_author,onehot_feat0,onehot_feat1,onehot_feat2,onehot_feat3,onehot_feat4,onehot_feat5,onehot_feat6,...,onehot_feat8,onehot_feat9,onehot_feat10,onehot_feat11,onehot_feat12,onehot_feat13,onehot_feat14,onehot_feat15,onehot_feat16,onehot_feat17
0,0,0,0,0,1,17,638,2,0,1,...,184,6,3,0,0,0,0,0,0,0
1,0,0,0,0,3,25,1021,0,0,1,...,186,6,2,0,0,0,0,0,0,0
2,0,0,0,0,6,8,402,0,0,0,...,51,2,3,0,0,0,0,0,0,0


,video_duration,video_width,video_height,music_id,video_tag_id,show_cnt,show_user_num,play_cnt,play_user_num,play_duration,...,21,22,23,24,25,26,27,28,29,30
0,5966.0,720,1280,3350323409,8,3710,2649,2213,1635,19547072,...,0,0,0,0,0,0,0,0,0,0
1,-1.0,886,1015,1812462382,27,30,25,8,7,94830,...,0,0,0,0,0,0,1,0,0,0
2,8000.0,720,1280,0,9,72,59,17,16,212893,...,0,0,0,0,0,0,0,0,0,0


## Dataset preparation

In [12]:
INTERACTION_N = 4_000_000

In [13]:
interaction_matrix = small_matrix.iloc[:INTERACTION_N][["user_id", "video_id", "watch_ratio"]].copy()

(user_ids, item_ids) = (interaction_matrix["user_id"].unique(), interaction_matrix["video_id"].unique())

# restrict the mappings to the unique user and item IDs
user_features_map = user_features_map.iloc[user_ids]
item_features_map = item_features_map.iloc[item_ids]


# Step 3: Model architecture

We will define the recommendation function based on a user_id.


### Items similarity matrix

In [44]:
item_similarity = cosine_similarity(item_features_map[item_features_columns])

# Fill the diagonal with -1 to not recommend.
np.fill_diagonal(item_similarity, -1)

item_similarity = pd.DataFrame(
    item_similarity,
    index=item_features_map.index,
    columns=item_features_map.index
)

item_similarity[:3]


,148,183,3649,5262,8234,6789,1963,175,1973,171,...,6523,5639,9820,2627,7270,2580,9165,2284,7126,9129
148,-1.000000,0.999962,0.999993,0.953802,0.204389,0.997641,0.999965,0.827857,0.999969,0.999993,...,0.237282,0.009100,0.999964,0.982976,0.009032,0.997511,0.521694,0.995519,0.009469,0.972857
183,0.999962,-1.000000,0.999988,0.951152,0.195874,0.997006,1.000000,0.822982,1.000000,0.999924,...,0.228823,0.000401,1.000000,0.981340,0.000334,0.996860,0.514253,0.994659,0.000770,0.970807
3649,0.999993,0.999988,-1.000000,0.952640,0.200630,0.997370,0.999990,0.825712,0.999992,0.999972,...,0.233548,0.005258,0.999989,0.982262,0.005190,0.997233,0.518413,0.995148,0.005627,0.971960


### User similarity matrix

In [45]:
user_similarity = cosine_similarity(user_features_map[user_features_columns])

# Fill the diagonal with -1 to not recommend.
np.fill_diagonal(user_similarity, -1)

user_similarity = pd.DataFrame(
    user_similarity,
    index=user_features_map.index,
    columns=user_features_map.index
)

user_similarity[:3]


,14,19,21,23,24,36,37,41,51,55,...,6275,6278,6283,6291,6296,6301,6306,6330,6331,6336
14,-1.000000,0.968534,0.989520,0.789409,0.868667,0.960966,0.975742,0.957665,0.972221,0.942045,...,0.820762,0.842183,0.881361,0.997691,0.943469,0.935506,0.906003,0.890153,0.942255,0.855118
19,0.968534,-1.000000,0.994194,0.614123,0.964449,0.999575,0.998531,0.999074,0.888465,0.995814,...,0.937009,0.949721,0.971112,0.975090,0.996216,0.993740,0.982740,0.975386,0.995905,0.957144
21,0.989520,0.994194,-1.000000,0.695077,0.930609,0.990743,0.996923,0.988975,0.931487,0.980536,...,0.894324,0.910746,0.939989,0.992973,0.981363,0.976701,0.957310,0.946547,0.980660,0.920854


In [69]:
def similar(id : int, similarity_matrix: pd.DataFrame, n : int = 10) -> pd.DataFrame:
    """
    Get the most similar objects to the given id from the similarity matrix.

    Args:
        id (int): The id of the object to find similar objects for.
        similarity_matrix (pd.DataFrame): The similarity matrix (user or item).
        n (int): The number of similar objects to return.

    Returns:
        pd.Index: The indices of the most similar objects.
    """

    # Retrieve column of similarity for the id
    try:
        similars = similarity_matrix[id]
    except KeyError:
        print(f"The {id} was not found in the similarity matrix.")
        print(f"You can try with an id between {similarity_matrix.index[:3].values}")
        return None

    # Get the index of the most similar objects
    index_similars = similars.nlargest(10).index.sort_values()

    return index_similars
    


In [75]:
interaction_matrix

,user_id,video_id,watch_ratio
0,14,148,0.722103
1,14,183,1.907377
2,14,3649,2.063311
3,14,5262,0.566388
4,14,8234,0.418364
...,...,...,...
4161590,6336,819,3.711897
4161591,6336,2953,0.575460
4161592,6336,10130,1.591007
4161593,6336,841,0.238834


In [76]:
def recommend_for_user(user_id: int, n: int = 10):
    """
    Recommend n videos for the user_id
    We expect user_similarity and item_similarity to be pre-computed
    """

    index_similars = similar(user_id, user_similarity, n)
    if index_similars is None:
        return None
    
    # Retrieve all interactions for similar users
    interactions_similar = interaction_matrix[interaction_matrix["user_id"].isin(index_similars)].copy()

    # Get the videos already seen by the target user
    user_seen_videos = interaction_matrix[interaction_matrix["user_id"] == user_id]["video_id"].unique()
    
    # Filter out already seen videos
    filtered = interactions_similar[~interactions_similar["video_id"].isin(user_seen_videos)]

    # Recommend top n videos with highest total watch_ratio
    recommended_videos = (
        filtered.groupby("video_id")["watch_ratio"]
        .sum()
        .sort_values(ascending=False)
        .head(n)
        .index
        .tolist()
    )

    return recommended_videos


# Step 4: Recommendation

- Predict which videos are likely to be enjoyed by each user in the test set
- Generate a top-N ranked list of recommendations for each user

In [ ]:
similar(14, user_similarity)

Index([396, 846, 885, 1161, 2057, 2526, 3821, 3987, 5505, 6014], dtype='int64')

In [ ]:
recommend_for_user(14, 10)

[3107, 4358, 2065, 694, 835, 3992, 7263, 9820, 152, 9926]

In [81]:
user_id = 14
recommendations = recommend_for_user(user_id, n=10)
print("Recommendations for user", user_id, ":", recommendations)

# To inspect what the user already saw:
seen = interaction_matrix[interaction_matrix["user_id"] == user_id]["video_id"].tolist()
print("Already watched:", seen)


Recommendations for user 14 : [3107, 4358, 2065, 694, 835, 3992, 7263, 9820, 152, 9926]
Already watched: [148, 183, 3649, 5262, 8234, 6789, 1963, 175, 1973, 171, 6803, 3634, 6787, 1951, 179, 5266, 5241, 6782, 6788, 8220, 6801, 3647, 6771, 9588, 186, 6812, 3684, 206, 211, 1988, 3672, 9595, 8242, 8248, 6829, 217, 9570, 139, 8160, 3669, 6846, 2007, 6839, 2000, 3654, 2008, 1898, 203, 5261, 256, 6854, 8289, 3702, 5326, 9660, 229, 5315, 262, 2040, 254, 2024, 5353, 6767, 8251, 8295, 5328, 8201, 9569, 2029, 223, 6865, 9653, 3706, 3630, 5331, 286, 2074, 2081, 3699, 9683, 1986, 2052, 285, 280, 5252, 8298, 5365, 9678, 275, 3719, 3586, 8212, 5367, 9592, 8319, 290, 145, 3737, 6904, 3722, 6749, 279, 147, 289, 5381, 1903, 9670, 8222, 296, 8316, 297, 2075, 2093, 9697, 2084, 3694, 3698, 5339, 2077, 258, 9645, 265, 2082, 1943, 5237, 3650, 5297, 8279, 3747, 307, 2113, 9659, 103, 5265, 8340, 5251, 6879, 3734, 288, 180, 5228, 5290, 8302, 8228, 8342, 6834, 5374, 9684, 5375, 3778, 6930, 9704, 340, 8323, 2121

# Step 5: Evaluation

- Choose suitable metrics (e.g., Precision@K, Recall@K, MAP, NDCG)
- Evaluate performance and provide interpretations

In [19]:
def calculate_precision_at_k(recommended_items, test_interactions, k=10):
    # Get the top K recommended item ids
    top_k_recommended = recommended_items[:k]

    # Check which of the top K items are in the test interactions (relevant items)
    relevant_items = test_interactions['video_id'].values
    precision = len(set(top_k_recommended) & set(relevant_items)) / k

    return precision


In [20]:
def calculate_recall_at_k(recommended_items, test_interactions, k=10):
    # Get the top K recommended item ids
    top_k_recommended = recommended_items[:k]

    # Check which of the top K items are in the test interactions (relevant items)
    relevant_items = test_interactions['video_id'].values
    recall = len(set(top_k_recommended) & set(relevant_items)) / len(relevant_items)

    return recall


In [21]:
# Example recommended items (Top 10)
recommended_items = [3586, 3589, 3590, 3595, 3597, 3599, 3607, 3608, 3610, 3615]

# Example test interactions for a user (user's actual interactions in the test set)
test_interactions = pd.DataFrame({
    'video_id': [3586, 3607, 3610, 3615],
    'watch_ratio': [1.0, 0.8, 0.9, 0.7]
})

# Calculate Precision@K and Recall@K
precision = calculate_precision_at_k(recommended_items, test_interactions, k=10)
recall = calculate_recall_at_k(recommended_items, test_interactions, k=10)

print(f"Precision@10: {precision:.2f}")
print(f"Recall@10: {recall:.2f}")


Precision@10: 0.40
Recall@10: 1.00
